## Multilevel bambi

# GENDER | COMPONENT

In [29]:
#| default_exp bambi_module

In [1]:
#| export
import pandas as pd
import bambi as bmb
import arviz as az
import numpy as np
from anthropmass.anthro_module import *

In [2]:
train=pd.read_csv('../data/processed/ANSURIIgendertexttrain.csv')

Setting priors for the  model

In [3]:
#| export
def setpriors_com(measurement, data):

    mean = data[measurement].mean()
    sd = data[measurement].std()

    mean_weight = data['weightkg'].mean()   
    mean_stature = data['stature'].mean()   

    priors = {
        "Intercept": bmb.Prior("Normal", mu=mean, sigma=sd*3),
        "Gender": bmb.Prior("Normal", mu=0, sigma=1),
        "weightkg": bmb.Prior("Normal", mu=mean/mean_weight, sigma=1),
        "stature": bmb.Prior("Normal", mu=mean/mean_stature, sigma=1),
        "Gender|Component": bmb.Prior("Normal", mu=0, sigma=bmb.Prior("Exponential", lam=1)),
        "sigma": bmb.Prior("Exponential", lam=1),
    }
    return priors
    

Multilevel model with Gender|Component

In [4]:
#| export
def component_model(measurement, data):
    model = bmb.Model(
    formula=f"{measurement} ~ 1 + Gender + weightkg + stature + (0 + Gender|Component)",
    data=data,
    priors=setpriors_com(measurement, data),
    noncentered=True
    )
    return model


In [5]:
def fitted_component_model(measurement, data):
    model=component_model(measurement,data)
    fitted_model = model.fit(draws=2000, tune=2000,target_accept=0.95, progressbar=True)

    return fitted_model

Making the model and fitting it and pickeling

In [ ]:
y_train=train.iloc[:,1:94].drop('weightkg',axis=1).drop('stature',axis=1)
variables = y_train.columns[0:5]
kindofmodel = 'bambi_ml_gc'
for var in variables:
    model = fitted_component_model(var, train)
    model.predict(model, kind="response")
    az.plot_ppc(model)
    make_pickle(model, kindofmodel, var)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, Gender, weightkg, stature, Gender|Component_sigma, Gender|Component_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 28 seconds.
There were 1 divergences after tuning. Increase `target_accept` or reparameterize.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, Gender, weightkg, stature, Gender|Component_sigma, Gender|Component_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 71 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, Gender, weightkg, stature, Gender|Component_sigma, Gender|Component_offset]


...

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 54 seconds.
There were 12 divergences after tuning. Increase `target_accept` or reparameterize.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, Gender, weightkg, stature, Gender|Component_sigma, Gender|Component_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 34 seconds.
There were 7 divergences after tuning. Increase `target_accept` or reparameterize.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, Gender, weightkg, stature, Gender|Component_sigma, Gender|Component_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 83 seconds.


In [7]:
model.predict(model, kind="response")
az.plot_ppc(model)

AttributeError: 'InferenceData' object has no attribute 'predict'

In [38]:
import nbdev; nbdev.nbdev_export()

JSONDecodeError: Expecting value: line 1 column 1 (char 0)